In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/account_features.csv")

print(df.shape)
print(df.columns.tolist())
print(df["churn_flag"].value_counts())

(500, 18)
['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag', 'tenure_days', 'current_mrr', 'total_usage_count', 'ticket_count', 'avg_satisfaction', 'escalation_rate', 'has_upgraded', 'has_downgraded']
churn_flag
False    390
True     110
Name: count, dtype: int64


In [2]:
for col in df.columns:
    print(col, "->", df[col].dtype)

account_id -> str
account_name -> str
industry -> str
country -> str
signup_date -> str
referral_source -> str
plan_tier -> str
seats -> int64
is_trial -> bool
churn_flag -> bool
tenure_days -> int64
current_mrr -> int64
total_usage_count -> int64
ticket_count -> float64
avg_satisfaction -> float64
escalation_rate -> float64
has_upgraded -> bool
has_downgraded -> bool


In [3]:
y = df["churn_flag"].astype(int)

In [4]:
features = [
    "industry",
    "country",
    "referral_source",
    "plan_tier",
    "seats",
    "is_trial",
    "tenure_days",
    "current_mrr",
    "total_usage_count",
    "ticket_count",
    "avg_satisfaction",
    "escalation_rate",
    "has_upgraded",
    "has_downgraded"
]

In [5]:
X = df[features]
y = df["churn_flag"].astype(int)

In [6]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (500, 14)
y shape: (500,)

Target distribution:
churn_flag
0    390
1    110
Name: count, dtype: int64


In [7]:
stratify=y

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTest target:")
print(y_test.value_counts())

Training set: (400, 14)
Test set: (100, 14)

Training target:
churn_flag
0    312
1     88
Name: count, dtype: int64

Test target:
churn_flag
0    78
1    22
Name: count, dtype: int64


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

categorical_features = [
    "industry",
    "country",
    "referral_source",
    "plan_tier"
]

numeric_features = [
    "seats",
    "is_trial",
    "tenure_days",
    "current_mrr",
    "total_usage_count",
    "ticket_count",
    "avg_satisfaction",
    "escalation_rate",
    "has_upgraded",
    "has_downgraded"
]

In [10]:
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("cat", categorical_transformer, categorical_features),
    ("num", numeric_transformer, numeric_features)
])

In [11]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [12]:
model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [13]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Predictions:", y_pred[:10])
print("Probabilities:", y_prob[:10])

Predictions: [0 0 0 0 0 0 0 0 0 0]
Probabilities: [0.0568505  0.05042558 0.40874106 0.14861016 0.23153098 0.09199699
 0.19174324 0.19771519 0.07906747 0.13802551]


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.78
Precision: 0.5
Recall   : 0.045454545454545456
F1 Score : 0.08333333333333333
ROC-AUC  : 0.6188811188811189

Confusion Matrix:
[[77  1]
 [21  1]]


In [15]:
baseline_results = {
    "model": "Logistic Regression",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob)
}

print(baseline_results)

{'model': 'Logistic Regression', 'accuracy': 0.78, 'precision': 0.5, 'recall': 0.045454545454545456, 'f1': 0.08333333333333333, 'roc_auc': 0.6188811188811189}


In [17]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [18]:

from xgboost import XGBClassifier

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
        eval_metric="logloss"
    ))
])

In [19]:
xgb_model.fit(X_train, y_train)

print("XGBoost training completed!")


XGBoost training completed!


In [20]:
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Predictions:", xgb_pred[:10])
print("Probabilities:", xgb_prob[:10])

Predictions: [0 0 1 0 0 0 0 0 0 0]
Probabilities: [0.12683378 0.01520424 0.7146337  0.15652195 0.37770095 0.05186981
 0.18780555 0.36970866 0.02387863 0.16668598]


In [21]:
print("Accuracy :", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred))
print("Recall   :", recall_score(y_test, xgb_pred))
print("F1 Score :", f1_score(y_test, xgb_pred))
print("ROC-AUC  :", roc_auc_score(y_test, xgb_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

Accuracy : 0.76
Precision: 0.3333333333333333
Recall   : 0.09090909090909091
F1 Score : 0.14285714285714285
ROC-AUC  : 0.5262237762237761

Confusion Matrix:
[[74  4]
 [20  2]]


In [22]:
print(df.groupby("churn_flag")[
    ["tenure_days",
     "current_mrr",
     "total_usage_count",
     "ticket_count",
     "avg_satisfaction",
     "escalation_rate",
     "has_upgraded",
     "has_downgraded"]
].mean())

            tenure_days   current_mrr  total_usage_count  ticket_count  \
churn_flag                                                               
False        329.433333  20734.500000         495.130769      4.020513   
True         371.672727  18846.845455         522.036364      3.927273   

            avg_satisfaction  escalation_rate  has_upgraded  has_downgraded  
churn_flag                                                                   
False               3.953395         0.043356      0.623077        0.358974  
True                4.002591         0.061017      0.581818        0.336364  


In [23]:
for col in ["industry", "country", "referral_source", "plan_tier"]:
    print(f"\n--- {col} ---")
    print(
        df.groupby(col)["churn_flag"]
        .agg(["count", "mean"])
        .sort_values("mean", ascending=False)
    )


--- industry ---
               count      mean
industry                      
DevTools         113  0.309735
FinTech          112  0.223214
HealthTech        96  0.218750
EdTech            79  0.164557
Cybersecurity    100  0.160000

--- country ---
         count      mean
country                 
DE          25  0.320000
US         291  0.233677
FR          22  0.227273
IN          49  0.204082
UK          58  0.189655
CA          23  0.173913
AU          32  0.125000

--- referral_source ---
                 count      mean
referral_source                 
event               96  0.302083
other              103  0.242718
ads                 98  0.234694
organic            114  0.175439
partner             89  0.146067

--- plan_tier ---
            count      mean
plan_tier                  
Enterprise    154  0.220779
Basic         168  0.220238
Pro           178  0.219101


In [24]:
df["satisfaction_missing"] = df["avg_satisfaction"].isna().astype(int)

print(
    df.groupby("satisfaction_missing")["churn_flag"]
    .agg(["count", "mean"])
)

                      count      mean
satisfaction_missing                 
0                       466  0.218884
1                        34  0.235294


In [25]:
churn_events = pd.read_csv(
    "../data/raw/ravenstack_churn_events.csv"
)

print(churn_events.columns.tolist())
print(churn_events.head())

['churn_event_id', 'account_id', 'churn_date', 'reason_code', 'refund_amount_usd', 'preceding_upgrade_flag', 'preceding_downgrade_flag', 'is_reactivation', 'feedback_text']
  churn_event_id account_id  churn_date reason_code  refund_amount_usd  \
0       C-816288   A-c37cab  2024-10-27     pricing               4.03   
1       C-5a81e7   A-37f969  2024-06-25     support              96.45   
2       C-a174be   A-b07346  2024-11-12      budget               0.00   
3       C-accb39   A-1e50e0  2023-11-01      budget              54.94   
4       C-92f889   A-956988  2024-12-30     unknown               0.00   

   preceding_upgrade_flag  preceding_downgrade_flag  is_reactivation  \
0                   False                     False            False   
1                    True                     False            False   
2                   False                     False            False   
3                   False                     False            False   
4                   Fa

In [26]:
subscriptions = pd.read_csv(
    "../data/raw/ravenstack_subscriptions.csv"
)

subscriptions["start_date"] = pd.to_datetime(
    subscriptions["start_date"]
)

subscriptions["end_date"] = pd.to_datetime(
    subscriptions["end_date"]
)

churn_events["churn_date"] = pd.to_datetime(
    churn_events["churn_date"]
)

print("Subscription date range:")
print(subscriptions["start_date"].min(), "to", subscriptions["start_date"].max())

print("\nChurn date range:")
print(churn_events["churn_date"].min(), "to", churn_events["churn_date"].max())

Subscription date range:
2023-01-09 00:00:00 to 2024-12-31 00:00:00

Churn date range:
2023-01-25 00:00:00 to 2024-12-31 00:00:00


In [28]:
usage = pd.read_csv(
    "../data/raw/ravenstack_feature_usage.csv"
)

subscriptions = pd.read_csv(
    "../data/raw/ravenstack_subscriptions.csv"
)

usage["usage_date"] = pd.to_datetime(usage["usage_date"])
churn_events["churn_date"] = pd.to_datetime(churn_events["churn_date"])

# Add account_id to usage through subscription_id
usage_with_account = usage.merge(
    subscriptions[["subscription_id", "account_id"]],
    on="subscription_id",
    how="left"
)

# Now connect usage with churn dates
usage_check = usage_with_account.merge(
    churn_events[["account_id", "churn_date"]],
    on="account_id",
    how="inner"
)

after_churn = usage_check[
    usage_check["usage_date"] > usage_check["churn_date"]
]

print("Usage records for churned accounts:", len(usage_check))
print("Usage records AFTER churn:", len(after_churn))

Usage records for churned accounts: 29891
Usage records AFTER churn: 6840


In [29]:
support = pd.read_csv(
    "../data/raw/ravenstack_support_tickets.csv"
)

support["submitted_at"] = pd.to_datetime(support["submitted_at"])

support_check = support.merge(
    churn_events[["account_id", "churn_date"]],
    on="account_id",
    how="inner"
)

after_churn_support = support_check[
    support_check["submitted_at"] > support_check["churn_date"]
]

print("Support records for churned accounts:", len(support_check))
print("Support records AFTER churn:", len(after_churn_support))

Support records for churned accounts: 2389
Support records AFTER churn: 556


In [30]:
print(subscriptions.columns.tolist())
print("\nSample:")
print(subscriptions.head())

['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier', 'seats', 'mrr_amount', 'arr_amount', 'is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag']

Sample:
  subscription_id account_id  start_date    end_date   plan_tier  seats  \
0        S-8cec59   A-3c1a3f  2023-12-23  2024-04-12  Enterprise     14   
1        S-0f6f44   A-9b9fe9  2024-06-11         NaN         Pro     17   
2        S-51c0d1   A-659280  2024-11-25         NaN  Enterprise     62   
3        S-f81687   A-e7a1e2  2024-11-23  2024-12-13  Enterprise      5   
4        S-cff5a2   A-ba6516  2024-01-10         NaN  Enterprise     27   

   mrr_amount  arr_amount  is_trial  upgrade_flag  downgrade_flag  churn_flag  \
0        2786       33432     False         False           False        True   
1         833        9996     False         False           False       False   
2           0           0      True          True           False       False   
3     

In [31]:
cutoff_date = pd.Timestamp("2024-09-30")
prediction_end = cutoff_date + pd.Timedelta(days=90)

print("Prediction cutoff:", cutoff_date.date())
print("Prediction window ends:", prediction_end.date())

future_churns = churn_events[
    (churn_events["churn_date"] > cutoff_date) &
    (churn_events["churn_date"] <= prediction_end)
]

print("\nChurn events in prediction window:", len(future_churns))
print("Unique accounts:", future_churns["account_id"].nunique())

Prediction cutoff: 2024-09-30
Prediction window ends: 2024-12-29

Churn events in prediction window: 240
Unique accounts: 175
